# CIP-Clay Free Energy: Extra Analysis
## Umbrella Integration | Thermodynamic Decomposition | Convergence | 2-D PMF & MFEP

Covers four analysis modules built on top of the base 1-D WHAM (`ClayPMF`):

| Section | Module | What it does |
|---------|--------|--------------|
| 3 | `ClayMeanForce` | Non-WHAM PMF via umbrella integration (RFD / RBF / GPR) |
| 4 | `ClayThermo` | Thermodynamic decomposition DH(r) from per-window EDR |
| 5 | `ClayConvergence` | Block-split & cumulative WHAM convergence diagnostics |
| 6-7 | `ClayPMF2D` + `ClayPath` | 2-D W(r,theta) surface & string-method MFEP |

**Usage**: set `UMBRELLA_DIR` in **Section 1**, then run all cells in order.

---
*Author: R.Swai — 2026*

In [ ]:
# Run once at the start of your session
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys, glob
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, '/Users/roev0007/Documents/solvation_shells/my_scripts')

from ClayPMF         import ClayPMF
from ClayPMF2D       import ClayPMF2D
from ClayMeanForce   import ClayMeanForce
from ClayThermo      import ClayThermo
from ClayConvergence import ClayConvergence
from ClayPath        import ClayPath
from ClayPMFPlotter  import ClayPMFPlotter

%matplotlib inline
plt.rcParams['figure.dpi']    = 100
plt.rcParams['savefig.dpi']   = 300
plt.rcParams['font.size']     = 11
plt.rcParams['axes.titlesize'] = 12

## 1. System Setup - Paths & Parameters

In [ ]:
# ==========================================================================
#  Edit UMBRELLA_DIR to the folder containing your pullx*.xvg files
#
#  Example path (Positive CIP+, NaCl, replicate 1, 11 ion pairs):
#    /Volumes/My_bckp/project_1_US/US/CIP/with_salts/Positive_with_salt
#    /CIP_NaCl_KCl/NaCl/kReplicate1/CIP+_the_side_cross_NaCl
#    /US_2CIP+_..._nosalt_11NaCl_50ns/Umbrella
# ==========================================================================
UMBRELLA_DIR = (
    '/Volumes/My_bckp/project_1_US/US/CIP/with_salts/Positive_with_salt'
    '/CIP_NaCl_KCl/NaCl/kReplicate1'
    '/CIP+_the_side_cross_NaCl'
    '/US_2CIP+_gaff_clayff_spc_mmt_2021_k1000_d0.1_PULLNVT_Constr_Only_at_start_nosalt_11NaCl_50ns'
    '/Umbrella'
)

K_SPRING       = 1000.0   # kJ mol-1 nm-2
TEMPERATURE    = 298.0    # K
EQUIL_SKIP     = 1000.0   # ps  (first 1 ns discarded)
N_BINS         = 200
XI_MIN         = None     # auto-detect
XI_MAX         = None     # auto-detect
TOLERANCE      = 1e-6
MAX_ITER       = 50000
CLAY_SELECTION = 'resname MMT'

OUTPUT_DIR = os.path.join(os.path.dirname(UMBRELLA_DIR), 'ExtraAnalysis_results')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Umbrella dir : {UMBRELLA_DIR}')
print(f'Output dir   : {OUTPUT_DIR}')

In [ ]:
print(f'Windows  : {N_WINDOWS}')
print(f'  First  : {os.path.basename(_pullx_files[0])}')
print(f'  Last   : {os.path.basename(_pullx_files[-1])}')

_gro_candidates = (sorted(glob.glob(os.path.join(UMBRELLA_DIR, 'umbrella*.gro')))
                   or sorted(glob.glob(os.path.join(UMBRELLA_DIR, 'preumbrella*.gro'))))
GRO_FILE = _gro_candidates[0] if _gro_candidates else None
print(f'GRO file : {GRO_FILE or "not found"}')

# TPR preferred for theta analysis (has bonds → unwrap() works correctly)
_tpr_candidates = (sorted(glob.glob(os.path.join(UMBRELLA_DIR, 'umbrella*.tpr')))
                   or sorted(glob.glob(os.path.join(UMBRELLA_DIR, '*.tpr'))))
TPR_FILE = _tpr_candidates[0] if _tpr_candidates else None
print(f'TPR file : {TPR_FILE or "not found (will fall back to GRO)"}')


## 2. Base 1-D WHAM - ClayPMF

Run the standard 1-D WHAM to produce W(r).  
The `pmf` object is passed to `ClayMeanForce` and `ClayConvergence` as baseline.

In [ ]:
pmf = ClayPMF(
    umbrella_dir  = UMBRELLA_DIR,
    n_windows     = N_WINDOWS,
    k             = K_SPRING,
    T             = TEMPERATURE,
    equil_skip_ps = EQUIL_SKIP,
    n_bins        = N_BINS,
    xi_min        = XI_MIN,
    xi_max        = XI_MAX,
    pullx_prefix  = 'pullx',
    tolerance     = TOLERANCE,
    max_iter      = MAX_ITER,
    verbose       = True,
)

pmf.load_data()
pmf.run_wham()

if GRO_FILE:
    pmf.detect_clay_surface(GRO_FILE, clay_selection=CLAY_SELECTION)

print(f'PMF range : [{pmf.pmf_abs.min():.2f}, {pmf.pmf_abs.max():.2f}] kJ/mol')
print(f'Bins      : {len(pmf.bin_centers_abs)}  dr = {pmf.bin_width:.4f} nm')

In [ ]:
if GRO_FILE:
    try:
        z_surf = pmf.detect_clay_surface(
            structure_file = GRO_FILE,
            clay_selection = CLAY_SELECTION,
        )
        print(f'Clay surface z = {z_surf:.4f} nm')
    except Exception as e:
        print(f'detect_clay_surface failed: {e}')
else:
    print('No GRO file found - surface detection skipped')

## 3. Umbrella Integration PMF - ClayMeanForce

Integrates the mean force from `pullf*.xvg` files using three estimators:

| Estimator | Method |
|-----------|--------|
| **RFD** | Regularised finite differences |
| **RBF** | Radial basis function spline (scipy) |
| **GPR** | Gaussian Process with Matern kernel (scikit-learn) |

GPR is silently skipped if `scikit-learn` is not installed.

In [ ]:
mf = ClayMeanForce(
    pmf3d          = pmf,
    pullf_prefix   = 'pullf',
    n_blocks       = 5,
    bulk_fraction  = 0.2,
    rbf_smoothing  = 0.1,
    gpr_n_restarts = 5,
)

mf.load_forces(use_pullf_files=True)
print(f'Windows  : {len(mf.r_eval)}')
print(f'r range  : [{mf.r_eval.min():.3f}, {mf.r_eval.max():.3f}] nm')
print(f'F range  : [{mf.mean_force.min():.1f}, {mf.mean_force.max():.1f}] kJ/(mol nm)')

In [ ]:
mf.run_rfd(n_grid=300)
mf.run_rbf(n_grid=300)

try:
    mf.run_gpr(n_grid=300)
except Exception as e:
    print(f'GPR skipped: {e}')

In [ ]:
mf.reference_to_bulk(bulk_fraction=0.2)

print('Adsorption energies (DG_ads = PMF_surface - PMF_bulk):')
for method in ('rfd', 'rbf', 'gpr'):
    try:
        dG = mf.adsorption_energy(method=method)
        print(f'  {method.upper():3s}: {dG:+.2f} kJ/mol')
    except Exception:
        pass

In [ ]:
plotter1d = ClayPMFPlotter(pmf=pmf)

fig_mf, ax_mf = plotter1d.plot_meanforce_vs_wham(
    clay_mf  = mf,
    figsize  = (9, 5),
    unit     = 'kJ/mol',
    show_gpr = True,
    title    = 'Umbrella Integration vs WHAM',
)

plotter1d.save_figure(fig_mf, os.path.join(OUTPUT_DIR, 'meanforce_vs_wham.png'))
plt.show()

In [ ]:
mf.print_summary()

In [ ]:
mf.save(os.path.join(OUTPUT_DIR, 'clay_meanforce.npz'))

## 4. Thermodynamic Decomposition - ClayThermo

Reads **raw `umbrella*.edr`** files (NOT rerun energy-group files) via `gmx energy`.

Outputs:
- **DH(r)** - per-window enthalpy relative to bulk
- Components: clay LJ/Coul, ion LJ/Coul, water LJ/Coul, intra-drug

> `gmx` must be on PATH or at `/usr/local/gromacs/bin/gmx`.

In [ ]:
# ClayThermo auto-detects n_windows from umbrella*.edr files - no n_windows arg
thermo = ClayThermo(
    umbrella_dir    = UMBRELLA_DIR,
    temperature     = TEMPERATURE,
    n_blocks        = 5,
    bulk_fraction   = 0.2,
    equil_skip_frac = 0.2,
    edr_prefix      = 'umbrella',
    pullx_prefix    = 'pullx',
)
print(f'ClayThermo: {thermo.n_windows} windows detected from EDR')

In [ ]:
thermo.load_energies(verbose=True)

In [ ]:
thermo.compute_enthalpy()
thermo.decompose()
print('Decomposition complete')
print(f'r range : [{thermo.r_centers.min():.3f}, {thermo.r_centers.max():.3f}] nm')
print(f'DH range: [{thermo.dH.min():.2f}, {thermo.dH.max():.2f}] kJ/mol')

In [ ]:
plotter_th = ClayPMFPlotter(pmf=pmf)

fig_th, axes_th = plotter_th.plot_thermo_decomposition(
    clay_thermo = thermo,
    figsize     = (11, 13),
    title       = 'Thermodynamic Decomposition  DH(r)',
)

plotter_th.save_figure(fig_th, os.path.join(OUTPUT_DIR, 'thermo_decomposition.png'))
plt.show()

In [ ]:
thermo.print_summary()

In [ ]:
thermo.save(os.path.join(OUTPUT_DIR, 'clay_thermo.npz'))

## 5. WHAM Convergence Analysis - ClayConvergence

Two diagnostics on the same `pullx*.xvg` data:

1. **Cumulative** - PMF from 0 to 10%, 20%, ..., 100% of each window's frames
2. **Block split** - PMF from N non-overlapping equal blocks

A converged simulation shows a flat cumulative PMF and narrow block spread.

In [ ]:
conv = ClayConvergence(
    umbrella_dir  = UMBRELLA_DIR,
    n_windows     = N_WINDOWS,
    k             = K_SPRING,
    T             = TEMPERATURE,
    equil_skip_ps = EQUIL_SKIP,
    n_bins        = N_BINS,
    pullx_prefix  = 'pullx',
    tolerance     = TOLERANCE,
    max_iter      = MAX_ITER,
    bulk_fraction = 0.2,
)

conv.run_full_wham(verbose=True)
ads = conv.adsorption_energies()
print(f"DG_ads (full) = {ads['full']:.2f} kJ/mol")

In [ ]:
conv.run_cumulative(n_checkpoints=5, verbose=True)
print(f'Cumulative checkpoints : {len(conv.cumulative_fracs)}')

conv.run_block_split(n_blocks=5, verbose=True)
print(f'Block PMFs computed    : {conv.n_blocks}')

drift = conv.drift_metric()
if drift is not None:
    print(f'Max drift (first->last): {drift.max():.3f} kJ/mol')
    print(f'Mean drift             : {drift.mean():.3f} kJ/mol')

In [ ]:
plotter_cv = ClayPMFPlotter(pmf=pmf)

fig_cv, axes_cv = plotter_cv.plot_pmf_convergence(
    clay_convergence = conv,
    show_cumulative  = True,
    show_blocks      = True,
    show_full_pmf    = True,
    title            = 'PMF Convergence',
)

plotter_cv.save_figure(fig_cv, os.path.join(OUTPUT_DIR, 'pmf_convergence.png'))
plt.show()

In [ ]:
fig_cm, ax_cm = plotter_cv.plot_convergence_metrics(
    clay_convergence = conv,
    figsize          = (9, 5),
    title            = 'Convergence Metrics',
)

plotter_cv.save_figure(fig_cm, os.path.join(OUTPUT_DIR, 'convergence_metrics.png'))
plt.show()

In [ ]:
conv.print_summary()

In [ ]:
conv.save(os.path.join(OUTPUT_DIR, 'clay_convergence.npz'))

## 6. 2-D Free Energy Surface - ClayPMF2D

W(r, theta) where:
- **r** = distance from clay surface [nm]
- **theta** = tilt angle of CIP long axis vs. clay normal [degrees]

theta is computed from XTC trajectories via MDAnalysis and cached for re-use.

In [ ]:
pmf2d = ClayPMF2D(
    umbrella_dir  = UMBRELLA_DIR,
    n_windows     = N_WINDOWS,
    k             = K_SPRING,
    T             = TEMPERATURE,
    equil_skip_ps = EQUIL_SKIP,
    n_r_bins      = 50,
    n_theta_bins  = 36,
    theta_range   = (0.0, 90.0),
    pullx_prefix  = 'pullx',
    tolerance     = TOLERANCE,
    max_iter      = MAX_ITER,
    verbose       = True,
)

pmf2d.load_data()
print(f'2D data loaded: {pmf2d.n_windows} windows')

In [ ]:
if GRO_FILE:
    try:
        z_surf_2d = pmf2d.detect_clay_surface(
            structure_file = GRO_FILE,
            clay_selection = CLAY_SELECTION,
        )
        print(f'Clay surface z = {z_surf_2d:.4f} nm')
    except Exception as e:
        print(f'detect_clay_surface failed: {e}')
else:
    print('No GRO file - surface detection skipped')

In [ ]:
# MDAnalysis selection for ring atoms of BOTH CIP molecules
# Adjust resname / atom names to match your topology
CIP_RING_SEL = 'resname api and (name N6 C10 C11 C12 C19 C21 C22 C23 C4 C5)'

_xtc_files = sorted(glob.glob(os.path.join(UMBRELLA_DIR, 'umbrella*.xtc')))
print(f'Found {len(_xtc_files)} XTC files')

if len(_xtc_files) == N_WINDOWS and GRO_FILE:
    pmf2d.load_theta_data(
        traj_files    = _xtc_files,
        topology      = GRO_FILE,
        cip_selection = CIP_RING_SEL,
        stride        = 1,
        cache_file    = os.path.join(OUTPUT_DIR, 'theta_cache.npz'),
        save_cache    = True,
    )
    print('Theta data loaded.')
elif len(_xtc_files) != N_WINDOWS:
    print(f'WARNING: {len(_xtc_files)} XTC files found but {N_WINDOWS} expected')
else:
    print('WARNING: No GRO file - cannot load theta data')

In [ ]:
pmf2d.run_wham_2d()
pmf2d.reference_to_bulk(bulk_fraction=0.2, enabled=True)

finite = pmf2d.pmf_2d[np.isfinite(pmf2d.pmf_2d)]
print(f'2D WHAM done   shape = {pmf2d.pmf_2d.shape}')
print(f'r     : [{pmf2d.r_centers.min():.3f}, {pmf2d.r_centers.max():.3f}] nm')
print(f'theta : [{pmf2d.theta_centers.min():.1f}, {pmf2d.theta_centers.max():.1f}] deg')
print(f'PMF   : [{finite.min():.2f}, {finite.max():.2f}] kJ/mol')

In [ ]:
plotter2d = ClayPMFPlotter(pmf2d=pmf2d)

fig_2d_ov = plotter2d.plot_2d_overview(
    figsize  = (14, 10),
    unit     = 'kJ/mol',
    zero_at  = 'bulk',
    suptitle = '2-D Free Energy  W(r, theta)',
)

plotter2d.save_figure(fig_2d_ov, os.path.join(OUTPUT_DIR, 'pmf2d_overview.png'))
plt.show()

In [ ]:
pmf2d.save_results(outdir=OUTPUT_DIR, prefix='pmf2d')
print(f'ClayPMF2D results saved to {OUTPUT_DIR}')

## 7. Minimum Free Energy Path - ClayPath

The **string method** finds the MFEP on W(r, theta) connecting the
adsorbed and bulk minima.

Key outputs:
- `r_path`, `theta_path` - MFEP coordinates [nm, degrees]
- `activation_energy()` - forward / reverse DG-dagger
- `adsorption_energy()` - end-to-end free energy drop ~ DG_ads

> `auto_endpoints(mode='adsorption')` places start at the deepest adsorption
> minimum and end at the bulk (last grid row).

In [ ]:
clay_path = ClayPath(
    pmf2d       = pmf2d,
    n_images    = 60,
    nan_penalty = 50.0,
)

clay_path.auto_endpoints(mode='adsorption')
print(f'Start : r = {clay_path.r_start:.3f} nm,  theta = {clay_path.theta_start:.1f} deg')
print(f'End   : r = {clay_path.r_end:.3f} nm,    theta = {clay_path.theta_end:.1f} deg')

In [ ]:
# Optional: manually override endpoints if auto_endpoints chooses poorly
# clay_path.set_endpoints(
#     r_start=0.30, theta_start=45.0,   # adsorbed end [nm, deg]
#     r_end=0.80,   theta_end=45.0,     # bulk end
# )

In [ ]:
clay_path.run_string_method(
    step_size = 0.02,
    max_iter  = 3000,
    tol       = 1e-5,
    verbose   = True,
)

clay_path.print_summary()

In [ ]:
fig_mfep2d, ax_mfep2d = plotter2d.plot_mfep_2d(
    clay_path   = clay_path,
    x_coord     = 'dist',
    cmap        = 'viridis',
    vmax        = 10.0,
    path_color  = 'white',
    path_lw     = 2.5,
    show_saddle = True,
    title       = 'MFEP on W(r, theta)',
)

plotter2d.save_figure(fig_mfep2d, os.path.join(OUTPUT_DIR, 'mfep_2d.png'))
plt.show()

In [ ]:
fig_mfep1d, ax_mfep1d = plotter2d.plot_mfep_profile(
    clay_path   = clay_path,
    x_axis      = 'arc_length',
    fill_below  = True,
    show_saddle = True,
    title       = 'PMF along MFEP',
)

plotter2d.save_figure(fig_mfep1d, os.path.join(OUTPUT_DIR, 'mfep_profile.png'))
plt.show()

In [ ]:
act      = clay_path.activation_energy()
ads_mfep = clay_path.adsorption_energy()
saddle   = clay_path.saddle_point()

print('MFEP energetics:')
print(f"  DG_fwd (adsorbed->saddle) = {act['forward']:+.2f} kJ/mol")
print(f"  DG_rev (bulk->saddle)     = {act['reverse']:+.2f} kJ/mol")
print(f"  DG_ads (end-to-end)       = {ads_mfep:+.2f} kJ/mol")
print(f"  Saddle: r={saddle['r']:.3f} nm  theta={saddle['theta']:.1f} deg  W={saddle['pmf']:.2f} kJ/mol")

In [ ]:
clay_path.save(os.path.join(OUTPUT_DIR, 'clay_path.npz'))

---
## 9. MBAR — Multistate Bennett Acceptance Ratio

MBAR is an asymptotically optimal, bin-free estimator of umbrella sampling free energies (Shirts & Chodera 2008).  Unlike WHAM, MBAR does not require binning during optimisation; the 1-D W(r) PMF is recovered by reweighting all samples to the unbiased ensemble.

> **Prerequisite**: `pymbar >= 4.0` must be installed:
> ```
> conda install -c conda-forge pymbar
> ```
> If pymbar is not installed, `ClayMBAR.__init__` will raise an `ImportError` with instructions.

In [ ]:
from ClayMBAR import ClayMBAR

# Requires pmf.load_data() to have been called (run_wham is optional but
# recommended so that the WHAM curve is available for comparison).
# K = 2 * N_WINDOWS pseudo-states (both CIP molecules per window).
clay_mbar = ClayMBAR(
    pmf=pmf,
    n_bins=200,
    bulk_fraction=0.2,
)
print(clay_mbar)

# Run MBAR  (may take 30–120 s depending on dataset size)
clay_mbar.run_mbar(uncertainty_method='analytical', verbose=True)

In [ ]:
# Reference both WHAM and MBAR PMFs to the bulk plateau (high-r end).
clay_mbar.reference_to_bulk()

# Adsorption free energy from MBAR
dG_mbar, r_min_mbar = clay_mbar.adsorption_energy(r_surface=0.5)
print(f'MBAR  ΔG_ads = {dG_mbar:.2f} kJ/mol  (minimum at r = {r_min_mbar:.3f} nm)')

# Compare with WHAM adsorption energy (if available)
try:
    dG_wham, r_min_wham = pmf.get_adsorption_energy()
    print(f'WHAM  ΔG_ads = {dG_wham:.2f} kJ/mol  (minimum at r = {r_min_wham:.3f} nm)')
    print(f'Δ(MBAR−WHAM) = {dG_mbar - dG_wham:.2f} kJ/mol')
except Exception as e:
    print(f'WHAM adsorption energy: {e}')

In [ ]:
# MDAnalysis selection for the CIP aromatic (quinolone) ring atoms
CIP_RING_SEL = 'resname api and (name N6 C10 C11 C12 C19 C21 C22 C23 C4 C5)'

_xtc_files = sorted(glob.glob(os.path.join(UMBRELLA_DIR, 'umbrella*.xtc')))
print(f'Found {len(_xtc_files)} XTC files')

# Use TPR (has bonds → unwrap works); fall back to GRO if TPR not available
_top_for_theta = TPR_FILE or GRO_FILE

if len(_xtc_files) == N_WINDOWS and _top_for_theta:
    print(f'Topology for theta: {os.path.basename(_top_for_theta)}')
    pmf2d.load_theta_data(
        traj_files    = _xtc_files,
        topology      = _top_for_theta,
        cip_selection = CIP_RING_SEL,
        stride        = 1,
        cache_file    = os.path.join(OUTPUT_DIR, 'theta_cache.npz'),
        save_cache    = True,
    )
    print('Theta data loaded.')
elif len(_xtc_files) != N_WINDOWS:
    print(f'WARNING: {len(_xtc_files)} XTC files found but {N_WINDOWS} expected')
else:
    print('WARNING: No topology file found - cannot load theta data')


In [ ]:
clay_mbar.print_summary()

In [ ]:
clay_mbar.save(os.path.join(OUTPUT_DIR, 'clay_mbar.npz'))

### 9b. MBAR 2-D PMF  W(r, θ)

Using the MBAR weights already computed by `run_mbar()`, the samples are
reweighted onto a 2-D (r, θ) histogram to yield W(r, θ) without re-running
MBAR from scratch.

θ is the tilt angle of the CIP aromatic ring normal relative to the clay
surface normal (0° = flat-on; 90° = edge-on), computed from MD trajectories
by `ClayPMF2D.load_theta_data()`.

**Prerequisites:** `pmf2d.theta_data` must already be populated (Section 6)
and `clay_mbar.run_mbar()` must have been called in the current session so
that the in-memory sample arrays `_x_all` and `_u_kn` are available.

In [ ]:
# --- 9b-1. Verify theta data is loaded (Section 6) -----------------------
# theta_per_window comes from pmf2d.theta_data which is a list of
# (theta1_arr, theta2_arr) tuples — one tuple per umbrella window.
# It must have been populated by pmf2d.load_theta_data() in Section 6.

if pmf2d.theta_data is None:
    raise RuntimeError(
        "pmf2d.theta_data is None.  "
        "Run pmf2d.load_theta_data(...) in Section 6 first."
    )
print(f"theta_data available: {len(pmf2d.theta_data)} windows")
t1_ex, t2_ex = pmf2d.theta_data[0]
print(f"  Window 0: CIP1 θ shape={t1_ex.shape},  CIP2 θ shape={t2_ex.shape}")
print(f"  θ range (window 0): [{t1_ex.min():.1f}, {t1_ex.max():.1f}]°")

# --- 9b-2. Compute 2-D MBAR PMF  W(r, θ) ----------------------------------
# run_mbar_2d() uses the MBAR weights already stored in clay_mbar._u_kn
# and clay_mbar._N_k.  No second MBAR optimisation is needed.

clay_mbar.run_mbar_2d(
    theta_per_window=pmf2d.theta_data,
    n_r_bins=None,          # default: same as 1-D MBAR (self.n_bins)
    n_theta_bins=36,        # 2.5° resolution over 0–90°
    theta_range=(0.0, 90.0),
    bulk_fraction=0.2,      # high-r reference region
    verbose=True,
)

print("\n2-D MBAR PMF shape:", clay_mbar.pmf_mbar_2d.shape)
print(f"r    range: [{clay_mbar.r_centers_2d.min():.3f}, "
      f"{clay_mbar.r_centers_2d.max():.3f}] nm")
print(f"theta range: [{clay_mbar.theta_centers_2d.min():.1f}, "
      f"{clay_mbar.theta_centers_2d.max():.1f}]°")

import numpy as np
n_valid = int(np.sum(np.isfinite(clay_mbar.pmf_mbar_2d)))
print(f"Valid (finite) bins: {n_valid}/{clay_mbar.pmf_mbar_2d.size}")

# Save the extended object (includes both 1-D and 2-D results)
clay_mbar.save(os.path.join(OUTPUT_DIR, 'clay_mbar.npz'))
print("\nMBAR results (1-D + 2-D) saved.")

In [ ]:
# --- 9b-3. Visualise W(r, θ): MBAR vs WHAM side-by-side ------------------
# plot_mbar_2d() uses self.pmf2d for the WHAM surface (if available) and
# clay_mbar.pmf_mbar_2d for the MBAR surface.
# compare_wham=True (default) draws both panels on a shared colour scale.

fig_2d, axes_2d = plotter1d.plot_mbar_2d(
    clay_mbar=clay_mbar,
    compare_wham=True,      # side-by-side WHAM | MBAR
    unit='kJ/mol',
    cmap='viridis',
    levels=25,
    vmax=None,              # auto: 95th percentile of finite values
    smooth=True,
    smooth_sigma=0.8,
    show_minimum=True,
    figsize=(13, 5),
)
fig_2d.savefig(
    os.path.join(OUTPUT_DIR, 'mbar_2d_comparison.png'),
    dpi=300, bbox_inches='tight',
)
print("Saved: mbar_2d_comparison.png")

## 10. Summary of Saved Results

In [ ]:
print(f'Results saved in: {OUTPUT_DIR}')
print()
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {f:<45}  {size_kb:6.1f} kB')

---
## Reload Saved Objects

Uncomment any line below to reload a previously saved result without re-running the analysis.

In [ ]:
# mbar2   = ClayMBAR.load(os.path.join(OUTPUT_DIR, 'clay_mbar.npz'))
# mf2     = ClayMeanForce.load(os.path.join(OUTPUT_DIR, 'clay_meanforce.npz'))
# thermo2 = ClayThermo.load(os.path.join(OUTPUT_DIR, 'clay_thermo.npz'))
# conv2   = ClayConvergence.load(os.path.join(OUTPUT_DIR, 'clay_convergence.npz'))
# path2   = ClayPath.load(os.path.join(OUTPUT_DIR, 'clay_path.npz'))
print('Uncomment lines above to reload saved results.')